# 07 - Regional Comparison

This notebook asks whether **vulnerability is spread evenly across Israel**. Earlier stages
built the stop-level graph, listed its articulation points, and scored every stop with
centrality measures. Here we aggregate those per-stop results by **region** (North / Center /
South / Jerusalem, assigned from coordinates in stage 01) and by **metropolitan area**
(Tel Aviv / Haifa / Jerusalem / Be'er Sheva / Periphery), and compare how concentrated the
critical infrastructure is in each of them. The output is a single comparison table plus the
figures that go into the report.

**Research question answered here:** *Is the damage from removing key stops distributed
differently between the center of the country and the periphery?*

### Inputs (produced by earlier notebooks)
- `outputs/nb/04_centrality_analysis/tables/stop_metrics.csv` - one row per stop with
  `stop_id, stop_name, region, metro, lat, lon, degree, betweenness, ...`
- `outputs/nb/03_network_descriptive_analysis/tables/articulation_points.csv` - the stops whose
  removal disconnects the graph
- `outputs/nb/03_network_descriptive_analysis/tables/bridges.csv` - *optional*; if absent, the
  bridge columns are simply skipped

The exact folder names are not hard-coded: the loader searches the `outputs/nb` tree for the
file names and prefers a folder whose name starts with the expected stage number.

### Outputs (all under `outputs/nb/07_regional_comparison/`)
`tables/`
- `regional_summary.csv` - the main deliverable: one row per region
- `metro_summary.csv` - the same breakdown by metropolitan area
- `stops_with_region.csv` - every stop with the `is_ap` / `is_critical` flags used here
- `top_critical_by_region.csv` - the 5 highest-betweenness critical stops of each region
- `regional_significance.csv` - chi-square test of region vs. criticality

`figures/`
- `critical_stations_by_region.png`, `ap_and_bridges_by_region.png`,
  `avg_betweenness_by_region.png`, `stations_map_by_region.png`,
  `regional_vulnerability_comparison.png`, `metro_vulnerability_comparison.png`

**Runtime:** seconds. This notebook only aggregates CSVs that earlier stages already computed -
there is no graph algorithm running here.

## 1. Environment bootstrap

Locates the repository (cloning it when we are on Google Colab), makes the repo root the working
directory, and creates the shared `outputs/nb` folder. Every notebook in this project starts with
this identical cell so that the whole series runs unchanged both locally and in Colab.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. Libraries, tunable constants and the stage output folder

We install/import the scientific stack, then declare every knob of this analysis in one place so a
grader can change the definition of "critical" and re-run:

- `CRITICAL_QUANTILE = 0.90` - the top-10% betweenness cut-off used throughout the project.
- `MIN_REGION_N = 100` - any group with fewer stops than this is flagged as a small sample, because
  a percentage computed on a handful of stops is noise, not a finding.
- `WILSON_Z = 1.96` - z value for the 95% confidence interval we draw on every share.

The stage writes only into its own folder, `outputs/nb/07_regional_comparison/`.

In [ ]:
_ensure("pandas", "numpy", "matplotlib", "seaborn", "scipy")

import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)
pd.set_option("display.width", 160)

# ---- Tunables -------------------------------------------------------------------
CRITICAL_QUANTILE = 0.90   # "critical" = betweenness in the top 10% of the whole network
MIN_REGION_N      = 100    # below this many stops, a group is flagged as small-sample
WILSON_Z          = 1.96   # 95% confidence interval on every reported share

# ---- Stage folders --------------------------------------------------------------
STAGE   = OUT / "07_regional_comparison"
TABLES  = STAGE / "tables"
FIGURES = STAGE / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print("Stage folder:", STAGE)

## 3. Hebrew text in figures

The GTFS feed is Israeli, so stop names, region names and metro names are Hebrew. Matplotlib does
not apply the Unicode bidirectional algorithm, so Hebrew comes out reversed. We patch
`matplotlib.text.Text.set_text` once, before drawing anything. Region and metro names are also
translated to English for the figures (see the label maps below), but any value we did not
anticipate falls back to correctly-ordered Hebrew instead of mojibake.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. Consistent labels, colours and ordering

Small but important for a report: the same region must get the same colour in every figure. The
original script coloured bars by *position*, so after sorting a region changed colour between
charts - we fix that with an explicit name to colour map. Regions are always drawn in the fixed
order Center, North, South, Jerusalem; anything unexpected is appended at the end in grey.

In [ ]:
REGION_ORDER = ["מרכז", "צפון", "דרום", "ירושלים"]
REGION_EN = {"מרכז": "Center", "צפון": "North",
             "דרום": "South", "ירושלים": "Jerusalem"}
REGION_COLORS = {"מרכז": "#2563eb", "צפון": "#16a34a",
                 "דרום": "#dc2626", "ירושלים": "#d97706"}

METRO_EN = {"תל אביב": "Tel Aviv", "חיפה": "Haifa", "ירושלים": "Jerusalem",
            "באר שבע": "Be'er Sheva", "פריפריה": "Periphery"}
METRO_COLORS = {"תל אביב": "#2563eb", "חיפה": "#0891b2", "ירושלים": "#d97706",
                "באר שבע": "#dc2626", "פריפריה": "#6b7280"}

GREY = "#6b7280"

def region_label(name):
    """English label for a region, falling back to display-ordered Hebrew."""
    return REGION_EN.get(name, fix_he(name))

def metro_label(name):
    return METRO_EN.get(name, fix_he(name))

def region_color(name):
    return REGION_COLORS.get(name, GREY)

def metro_color(name):
    return METRO_COLORS.get(name, GREY)

def in_region_order(values):
    """Sort region names into the canonical order; unknown names go last, alphabetically."""
    known = [r for r in REGION_ORDER if r in set(values)]
    rest = sorted(v for v in set(values) if v not in REGION_ORDER)
    return known + rest

def annotate_bars(ax, bars, texts, pad=0.02, fontsize=9):
    """Write a short label (we use the group size n) just above each bar."""
    top = ax.get_ylim()[1]
    for bar, txt in zip(bars, texts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + top * pad,
                txt, ha="center", va="bottom", fontsize=fontsize, color="#334155")

print("label helpers ready")

## 5. Loading the upstream artifacts

This stage consumes results, it does not recompute them: the per-stop centrality table comes from
notebook 04 and the articulation-point list from notebook 03. `find_artifact` searches the whole
`outputs/nb` tree for a file name and prefers the folder whose name starts with the expected stage
number, so a rename upstream produces a helpful message instead of a `FileNotFoundError` deep in
the analysis. Our own stage folder is excluded from the search so a re-run can never read its own
output.

`bridges.csv` is treated as optional - it only adds one descriptive column.

In [ ]:
def find_artifact(filename, prefer_prefix, produced_by):
    """Locate a file written by an earlier notebook stage under outputs/nb."""
    candidates = sorted({p for p in OUT.rglob(filename) if STAGE not in p.parents})
    if not candidates:
        raise FileNotFoundError(
            f"{filename} not found anywhere under {OUT} - "
            f"run notebook {produced_by} first, it writes this file.")
    preferred = [p for p in candidates
                 if p.relative_to(OUT).parts[0].startswith(prefer_prefix)]
    chosen = (preferred or candidates)[0]
    if len(candidates) > 1:
        print(f"  note: found {len(candidates)} copies of {filename}; using {chosen}")
    return chosen

METRICS_PATH = find_artifact("stop_metrics.csv", "04", "04_centrality_analysis")
AP_PATH = find_artifact("articulation_points.csv", "03", "03_network_descriptive_analysis")

metrics = pd.read_csv(METRICS_PATH, dtype={"stop_id": str}, encoding="utf-8-sig")
ap_df = pd.read_csv(AP_PATH, dtype={"stop_id": str}, encoding="utf-8-sig")

try:
    BRIDGES_PATH = find_artifact("bridges.csv", "03", "03_network_descriptive_analysis")
    bridges = pd.read_csv(BRIDGES_PATH,
                          dtype={"from_stop": str, "to_stop": str},
                          encoding="utf-8-sig")
except FileNotFoundError:
    BRIDGES_PATH, bridges = None, None
    print("  bridges.csv not available - the bridge columns will be skipped.")

# Notebook 04 exports the sampled estimate under the name `approx_betweenness`
# (the name records that it is a k-sample estimate, not exact betweenness).
# Alias it so this notebook works with either schema.
if "betweenness" not in metrics.columns and "approx_betweenness" in metrics.columns:
    metrics["betweenness"] = metrics["approx_betweenness"]
    print("  note: aliased 'approx_betweenness' -> 'betweenness' (sampled estimate)")

# Fail loudly and early if the upstream schema is not what we expect.
for col in ("stop_id", "region", "metro", "degree", "betweenness", "lat", "lon"):
    if col not in metrics.columns:
        raise KeyError(f"'{col}' missing from {METRICS_PATH.name}. "
                       f"Columns present: {list(metrics.columns)}")

for col in ("degree", "betweenness", "lat", "lon"):
    metrics[col] = pd.to_numeric(metrics[col], errors="coerce")
metrics["degree"] = metrics["degree"].fillna(0)
metrics["betweenness"] = metrics["betweenness"].fillna(0)
for col in ("region", "metro"):
    metrics[col] = metrics[col].fillna("").replace("", "Unknown")

print(f"stops loaded      : {len(metrics):,}  <- {METRICS_PATH}")
print(f"articulation pts  : {len(ap_df):,}  <- {AP_PATH}")
print(f"bridges           : {0 if bridges is None else len(bridges):,}")
print("stops per region  :")
print(metrics["region"].value_counts().to_string())

## 6. The definition of "critical" used here

This is the single most important methodological choice of the notebook, so we state it explicitly.

> **A stop is *critical* if its betweenness centrality is in the top 10% of the entire network**
> (`betweenness >= p90`, threshold computed once over **all** stops, not per region).

This is exactly the definition used in **notebook 04**, where the same top-10% betweenness set is
tested for whether it has a walkable alternative. Keeping it identical means the counts in this
notebook add up to the counts there.

Two deliberate consequences:

- **The threshold is global.** If each region had its own p90, every region would be exactly 10%
  critical by construction and the comparison would be meaningless. A global threshold is what lets
  us say "region X holds more than its share of the network's critical stops".
- **Articulation points are reported separately, not merged in.** An articulation point is a
  *structural* failure point (removing it disconnects the graph), which is a different and stricter
  notion than high traffic load. The original script OR-ed the two together; we keep the primary
  `is_critical` flag aligned with notebook 04 and expose the union as a clearly-named secondary
  column `is_critical_broad`, so both readings are available without confusing them.

One caveat we check for in code: betweenness is computed on the largest connected component only,
so isolated stops score 0. If more than 10% of stops tie at the threshold value the `>=` comparison
silently selects more than 10% of the network - the cell prints the actual share so this is visible.

In [ ]:
ap_set = set(ap_df["stop_id"].astype(str))
metrics["is_ap"] = metrics["stop_id"].astype(str).isin(ap_set)

BTW_THRESHOLD = float(metrics["betweenness"].quantile(CRITICAL_QUANTILE))
metrics["is_critical"] = metrics["betweenness"] >= BTW_THRESHOLD
metrics["is_critical_broad"] = metrics["is_critical"] | metrics["is_ap"]

n_total = len(metrics)
n_crit = int(metrics["is_critical"].sum())
n_ap = int(metrics["is_ap"].sum())
n_both = int((metrics["is_critical"] & metrics["is_ap"]).sum())

print(f"betweenness p{CRITICAL_QUANTILE*100:.0f} threshold : {BTW_THRESHOLD:.8f}")
print(f"critical (top betweenness)   : {n_crit:,}  ({100*n_crit/n_total:.1f}% of all stops)")
print(f"articulation points matched  : {n_ap:,}  ({100*n_ap/n_total:.1f}%)")
print(f"both critical and AP         : {n_both:,}")
print(f"union (is_critical_broad)    : {int(metrics['is_critical_broad'].sum()):,}")

if BTW_THRESHOLD <= 0:
    print("\nWARNING: the p90 threshold is 0, meaning more than 10% of stops have zero "
          "betweenness. Every zero-betweenness stop is being counted as critical - "
          "raise CRITICAL_QUANTILE or restrict to the largest component before trusting this.")
if n_ap == 0:
    print("\nWARNING: no articulation point matched a stop_id in stop_metrics.csv - "
          "the two upstream files may use different id formats.")

## 7. Bridges touching each region

A *bridge* is an edge whose removal disconnects the graph - the edge equivalent of an articulation
point, and a direct measure of "this area hangs by a single line". The original script loaded
`bridges.csv` and then never used it; we actually use it here.

Counting rule: each bridge is counted **once per distinct region it touches**. A bridge internal to
one region adds 1 to that region; a bridge that crosses a regional boundary adds 1 to each of the
two. Endpoints whose stop is not in the metrics table (should not happen) are dropped. The cell is
a no-op if `bridges.csv` was not found.

In [ ]:
bridge_counts = None
if bridges is not None and len(bridges) and {"from_stop", "to_stop"} <= set(bridges.columns):
    region_of = metrics.drop_duplicates("stop_id").set_index("stop_id")["region"]
    touched = []
    for a, b in zip(bridges["from_stop"].map(region_of), bridges["to_stop"].map(region_of)):
        touched.extend({r for r in (a, b) if isinstance(r, str)})
    bridge_counts = pd.Series(touched, dtype="object").value_counts()
    print("bridges touching each region:")
    print(bridge_counts.to_string())
else:
    print("no bridge data - skipping the bridge columns")

## 8. The regional summary table

The core aggregation. For each region we report the raw counts (`total_stops`, `critical_stops`,
`ap_stops`), the averages (`avg_degree`, `avg_betweenness`, `max_betweenness`) and, most usefully
for comparison, the **shares** - a region with 13k stops and one with 3.7k stops cannot be compared
on counts alone.

Two additions over the original script:

- **A 95% Wilson confidence interval** on `pct_critical`. The Wilson interval behaves sensibly for
  small groups and for shares near 0 or 100%, where the textbook normal interval does not. It is
  what tells us whether two regions really differ or just look different.
- **A `small_sample` flag** (`total_stops < MIN_REGION_N`). Any flagged row is printed as a warning
  and hatched in the figures; its percentage should not be quoted in the report.

The same helper is reused for the metro breakdown in the next cell.

In [ ]:
def wilson_ci(k, n, z=WILSON_Z):
    """95% Wilson score interval (in percent) for k successes out of n."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (100 * max(0.0, center - half), 100 * min(1.0, center + half))

def summarize(df, key):
    """Aggregate stop-level flags into one row per group (region or metro)."""
    s = df.groupby(key).agg(
        total_stops=("stop_id", "count"),
        critical_stops=("is_critical", "sum"),
        critical_broad_stops=("is_critical_broad", "sum"),
        ap_stops=("is_ap", "sum"),
        avg_degree=("degree", "mean"),
        avg_betweenness=("betweenness", "mean"),
        max_betweenness=("betweenness", "max"),
    ).reset_index()
    for c in ("critical_stops", "critical_broad_stops", "ap_stops"):
        s[c] = s[c].astype(int)
    s["pct_critical"] = (100 * s["critical_stops"] / s["total_stops"]).round(2)
    s["pct_critical_broad"] = (100 * s["critical_broad_stops"] / s["total_stops"]).round(2)
    s["pct_ap"] = (100 * s["ap_stops"] / s["total_stops"]).round(2)
    # share of the network's critical stops that sit in this group
    s["share_of_all_critical"] = (100 * s["critical_stops"] / max(1, n_crit)).round(1)
    s["share_of_all_stops"] = (100 * s["total_stops"] / n_total).round(1)
    ci = [wilson_ci(k, n) for k, n in zip(s["critical_stops"], s["total_stops"])]
    s["pct_critical_lo"] = [round(lo, 2) for lo, _ in ci]
    s["pct_critical_hi"] = [round(hi, 2) for _, hi in ci]
    s["small_sample"] = s["total_stops"] < MIN_REGION_N
    s["avg_degree"] = s["avg_degree"].round(3)
    s["avg_betweenness"] = s["avg_betweenness"].round(8)
    return s.sort_values("pct_critical", ascending=False).reset_index(drop=True)

region_summary = summarize(metrics, "region")
if bridge_counts is not None:
    region_summary["bridges_touching"] = (region_summary["region"]
                                          .map(bridge_counts).fillna(0).astype(int))
    region_summary["bridges_per_1000_stops"] = (
        1000 * region_summary["bridges_touching"] / region_summary["total_stops"]).round(2)

region_summary.to_csv(TABLES / "regional_summary.csv", index=False, encoding="utf-8-sig")
metrics.to_csv(TABLES / "stops_with_region.csv", index=False, encoding="utf-8-sig")

print(region_summary.to_string(index=False))

flagged = region_summary[region_summary["small_sample"]]
if len(flagged):
    print(f"\nSMALL SAMPLE (< {MIN_REGION_N} stops) - do not quote these percentages:")
    print(flagged[["region", "total_stops", "pct_critical"]].to_string(index=False))
else:
    print(f"\nNo region falls below {MIN_REGION_N} stops - all regional shares are on "
          "thousands of stops and are statistically stable.")
print(f"\nsaved -> {TABLES / 'regional_summary.csv'}")

## 9. The same breakdown by metropolitan area

The four regions are geographic slabs cut by latitude, which lumps a dense city together with the
farmland around it. The `metro` label from stage 01 is the complementary view: stops within a fixed
radius of Tel Aviv / Haifa / Jerusalem / Be'er Sheva, everything else labelled Periphery. This is
where the small-sample flag can actually fire, so we print it again.

In [ ]:
metro_summary = summarize(metrics, "metro")
metro_summary.to_csv(TABLES / "metro_summary.csv", index=False, encoding="utf-8-sig")
print(metro_summary.to_string(index=False))

flagged_metro = metro_summary[metro_summary["small_sample"]]
if len(flagged_metro):
    print(f"\nSMALL SAMPLE (< {MIN_REGION_N} stops):")
    print(flagged_metro[["metro", "total_stops", "pct_critical",
                         "pct_critical_lo", "pct_critical_hi"]].to_string(index=False))
else:
    print(f"\nNo metro area falls below {MIN_REGION_N} stops.")
print(f"\nsaved -> {TABLES / 'metro_summary.csv'}")

## 10. Is the regional difference real, or just visible?

Bar charts always look different. A chi-square test of independence on the 2-way table
*region x is_critical* asks whether the share of critical stops depends on the region at all, and
**Cramer's V** turns the chi-square into an effect size between 0 and 1 that does not grow with the
sample size.

Read them together and be sceptical: with tens of thousands of stops, *any* difference comes out
"highly significant" (p is essentially a function of n here). Cramer's V and the confidence
intervals from the previous cell are the honest measures of how much the regions actually differ.

In [ ]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(metrics["region"], metrics["is_critical"])
chi2, p_value, dof, expected = chi2_contingency(contingency)
n_obs = int(contingency.values.sum())
cramers_v = float(np.sqrt(chi2 / (n_obs * (min(contingency.shape) - 1))))
spread = float(region_summary["pct_critical"].max() - region_summary["pct_critical"].min())

sig = pd.DataFrame([{
    "test": "chi-square independence (region x is_critical)",
    "chi2": round(chi2, 2),
    "dof": int(dof),
    "p_value": p_value,
    "n": n_obs,
    "cramers_v": round(cramers_v, 4),
    "pct_critical_spread_points": round(spread, 2),
    "min_expected_count": round(float(expected.min()), 1),
}])
sig.to_csv(TABLES / "regional_significance.csv", index=False, encoding="utf-8-sig")
print(sig.T.to_string(header=False))

print("\nInterpretation:")
print(f"  p = {p_value:.3g} -> the regions do differ, but at n={n_obs:,} that was almost "
      "guaranteed.")
print(f"  Cramer's V = {cramers_v:.3f} -> "
      + ("negligible" if cramers_v < 0.1 else "small" if cramers_v < 0.3
         else "moderate" if cramers_v < 0.5 else "large") + " association.")
print(f"  spread between the highest and lowest region: {spread:.1f} percentage points.")

## 11. Figure 1 - critical stops by region

Two panels, because they answer two different questions. **Left**: the absolute number of critical
stops, which is dominated by whichever region simply has the most stops. **Right**: the *share* of
each region's own stops that are critical, with the 95% Wilson interval as an error bar - this is
the panel that answers "is the periphery more fragile?". The group size `n` is printed above every
bar, and a small-sample region would be drawn hatched.

In [ ]:
reg = region_summary.set_index("region").loc[in_region_order(region_summary["region"])].reset_index()
labels = [region_label(r) for r in reg["region"]]
colors = [region_color(r) for r in reg["region"]]
hatches = ["//" if flag else "" for flag in reg["small_sample"]]
n_texts = [f"n={int(v):,}" for v in reg["total_stops"]]

yerr = np.vstack([
    np.clip(reg["pct_critical"] - reg["pct_critical_lo"], 0, None),
    np.clip(reg["pct_critical_hi"] - reg["pct_critical"], 0, None),
])

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

bars = axes[0].bar(labels, reg["critical_stops"], color=colors, edgecolor="white", hatch=hatches)
axes[0].set_title("Critical stops per region (count)")
axes[0].set_ylabel("number of critical stops")
axes[0].margins(y=0.15)
annotate_bars(axes[0], bars, n_texts)

bars2 = axes[1].bar(labels, reg["pct_critical"], color=colors, edgecolor="white",
                    hatch=hatches, yerr=yerr, capsize=5, ecolor="#334155")
axes[1].axhline(100 * n_crit / n_total, color="#334155", ls="--", lw=1.2,
                label=f"network average ({100*n_crit/n_total:.1f}%)")
axes[1].set_title("Share of the region's own stops that are critical")
axes[1].set_ylabel("% of the region's stops")
axes[1].margins(y=0.20)
axes[1].legend(fontsize=9)
annotate_bars(axes[1], bars2, n_texts)

fig.suptitle(f"Critical = betweenness in the network-wide top {100*(1-CRITICAL_QUANTILE):.0f}%"
             "   (error bars: 95% Wilson CI)", fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES / "critical_stations_by_region.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "critical_stations_by_region.png")

## 12. Figure 2 - structural failure points by region

Where figure 1 measured *load*, this one measures *structure*: the share of each region's stops that
are articulation points, and (when `bridges.csv` is available) how many bridges touch the region per
1,000 stops. A region can carry moderate traffic and still be fragile if its network is a chain of
single links, which is precisely what these two bars expose.

In [ ]:
has_bridges = "bridges_per_1000_stops" in reg.columns
ncols = 2 if has_bridges else 1
fig, axes = plt.subplots(1, ncols, figsize=(6.5 * ncols, 5))
axes = np.atleast_1d(axes)

b = axes[0].bar(labels, reg["pct_ap"], color=colors, edgecolor="white", hatch=hatches)
axes[0].set_title("Articulation points as % of the region's stops")
axes[0].set_ylabel("% of the region's stops")
axes[0].margins(y=0.15)
annotate_bars(axes[0], b, n_texts)

if has_bridges:
    b2 = axes[1].bar(labels, reg["bridges_per_1000_stops"], color=colors,
                     edgecolor="white", hatch=hatches)
    axes[1].set_title("Bridges touching the region per 1,000 stops")
    axes[1].set_ylabel("bridges per 1,000 stops")
    axes[1].margins(y=0.15)
    annotate_bars(axes[1], b2, n_texts)

plt.tight_layout()
plt.savefig(FIGURES / "ap_and_bridges_by_region.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "ap_and_bridges_by_region.png")

## 13. Figure 3 - how load is distributed inside each region

Averages hide shape, so we show two things. **Left**: mean betweenness per region - how much
shortest-path traffic a typical stop carries. **Right**: the distribution of betweenness within each
region on a log scale (zeros excluded, since log(0) is undefined), which reveals whether a region
has a few extreme hubs or a broad plateau. The dashed line is the global p90 threshold, i.e. the
cut-off above which a stop counts as critical.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

b = axes[0].bar(labels, reg["avg_betweenness"], color=colors, edgecolor="white", hatch=hatches)
axes[0].set_title("Mean betweenness centrality per region")
axes[0].set_ylabel("mean betweenness")
axes[0].margins(y=0.15)
annotate_bars(axes[0], b, n_texts)

order = list(reg["region"])
data = [metrics.loc[(metrics["region"] == r) & (metrics["betweenness"] > 0), "betweenness"].values
        for r in order]
parts = axes[1].boxplot(data, labels=labels, showfliers=False, patch_artist=True)
for patch, c in zip(parts["boxes"], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.65)
axes[1].set_yscale("log")
axes[1].axhline(max(BTW_THRESHOLD, 1e-12), color="#334155", ls="--", lw=1.2,
                label=f"critical threshold (p{CRITICAL_QUANTILE*100:.0f})")
axes[1].set_title("Betweenness distribution, non-zero stops only (log scale)")
axes[1].set_ylabel("betweenness")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES / "avg_betweenness_by_region.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "avg_betweenness_by_region.png")

## 14. Figure 4 - the geographic picture

Every stop plotted at its coordinates, coloured by region, with the critical stops overlaid in
black. This is a sanity check as much as a result: the latitude-band region definition from stage 01
should be visible as clean horizontal cuts, and the critical stops should trace the intercity
corridors rather than scatter randomly. Stops without coordinates are dropped, and the count of
dropped stops is printed.

In [ ]:
geo = metrics.dropna(subset=["lat", "lon"])
print(f"stops without coordinates (dropped from the map): {len(metrics) - len(geo):,}")

fig, ax = plt.subplots(figsize=(8, 11))
for r in in_region_order(geo["region"]):
    grp = geo[geo["region"] == r]
    ax.scatter(grp["lon"], grp["lat"], s=2, alpha=0.30, color=region_color(r),
               label=f"{region_label(r)} (n={len(grp):,})")

crit_geo = geo[geo["is_critical"]]
ax.scatter(crit_geo["lon"], crit_geo["lat"], s=14, color="black", alpha=0.70, zorder=5,
           label=f"critical (n={len(crit_geo):,})")

ax.set_title("Stops by region, critical stops in black")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_aspect(1 / np.cos(np.radians(float(geo["lat"].mean()))))
ax.legend(markerscale=4, fontsize=9, loc="upper left")
plt.tight_layout()
plt.savefig(FIGURES / "stations_map_by_region.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "stations_map_by_region.png")

## 15. Figure 5 - the four-panel regional comparison

The summary figure for the report: the four indicators side by side on the same set of regions -
% critical, % articulation points, mean degree (how well connected a typical stop is), and the share
of *the network's total* critical stops that sits in each region. The last panel is the
concentration view: a region holding a much larger share of the critical stops than of the stops
overall is where a national-scale disruption would hurt most, so we draw its share of all stops as a
reference marker.

In [ ]:
panels = [
    ("pct_critical", "% critical stops", "% of the region's stops"),
    ("pct_ap", "% articulation points", "% of the region's stops"),
    ("avg_degree", "Mean degree", "neighbouring stops"),
    ("share_of_all_critical", "Share of ALL critical stops", "% of the network's critical stops"),
]

fig, axes = plt.subplots(1, 4, figsize=(19, 5))
for ax, (col, title, ylab) in zip(axes, panels):
    bars = ax.bar(labels, reg[col], color=colors, edgecolor="white", hatch=hatches)
    ax.set_title(title)
    ax.set_ylabel(ylab)
    ax.margins(y=0.18)
    annotate_bars(ax, bars, n_texts, fontsize=8)
    if col == "share_of_all_critical":
        ax.scatter(labels, reg["share_of_all_stops"], color="black", marker="_", s=400,
                   zorder=6, label="share of all stops")
        ax.legend(fontsize=8)

fig.suptitle("Regional comparison of network vulnerability (hatched = small sample)", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / "regional_vulnerability_comparison.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "regional_vulnerability_comparison.png")

## 16. Figure 6 - the same comparison by metropolitan area

Repeats figure 5 for the metro breakdown, which is the sharper contrast: four dense city areas
against everything else ("Periphery"). Groups are sorted by their critical share, `n` is annotated,
and any group below `MIN_REGION_N` stops is hatched so a spuriously high percentage cannot be read
as a finding.

In [ ]:
mt = metro_summary.copy()
m_labels = [metro_label(m) for m in mt["metro"]]
m_colors = [metro_color(m) for m in mt["metro"]]
m_hatch = ["//" if f else "" for f in mt["small_sample"]]
m_texts = [f"n={int(v):,}" for v in mt["total_stops"]]

m_yerr = np.vstack([
    np.clip(mt["pct_critical"] - mt["pct_critical_lo"], 0, None),
    np.clip(mt["pct_critical_hi"] - mt["pct_critical"], 0, None),
])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

b0 = axes[0].bar(m_labels, mt["pct_critical"], color=m_colors, edgecolor="white",
                 hatch=m_hatch, yerr=m_yerr, capsize=4, ecolor="#334155")
axes[0].axhline(100 * n_crit / n_total, color="#334155", ls="--", lw=1.2)
axes[0].set_title("% critical stops (95% Wilson CI)")
axes[0].set_ylabel("% of the area's stops")
axes[0].margins(y=0.20)
annotate_bars(axes[0], b0, m_texts, fontsize=8)

b1 = axes[1].bar(m_labels, mt["pct_ap"], color=m_colors, edgecolor="white", hatch=m_hatch)
axes[1].set_title("% articulation points")
axes[1].set_ylabel("% of the area's stops")
axes[1].margins(y=0.18)
annotate_bars(axes[1], b1, m_texts, fontsize=8)

b2 = axes[2].bar(m_labels, mt["avg_degree"], color=m_colors, edgecolor="white", hatch=m_hatch)
axes[2].set_title("Mean degree")
axes[2].set_ylabel("neighbouring stops")
axes[2].margins(y=0.18)
annotate_bars(axes[2], b2, m_texts, fontsize=8)

for ax in axes:
    ax.tick_params(axis="x", rotation=20)

fig.suptitle("Metropolitan comparison (hatched = small sample)", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / "metro_vulnerability_comparison.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "metro_vulnerability_comparison.png")

## 17. Which stops actually drive each region's number

Percentages are abstract; naming the stops makes the result checkable against reality. For every
region we list its five highest-betweenness critical stops, together with whether each one is also
an articulation point. If these names are recognisable intercity hubs, the pipeline is measuring
something real.

In [ ]:
cols = ["region", "metro", "stop_id", "stop_name", "degree", "betweenness", "is_ap"]
cols = [c for c in cols if c in metrics.columns]

top_by_region = (metrics[metrics["is_critical"]]
                 .sort_values("betweenness", ascending=False)
                 .groupby("region", group_keys=False)
                 .head(5)[cols]
                 .sort_values(["region", "betweenness"], ascending=[True, False]))

top_by_region.to_csv(TABLES / "top_critical_by_region.csv", index=False, encoding="utf-8-sig")
print(top_by_region.to_string(index=False))
print(f"\nsaved -> {TABLES / 'top_critical_by_region.csv'}")

print("\nAll artifacts written by this notebook:")
for p in sorted(STAGE.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(OUT))

## Takeaways

*(The exact numbers are printed by the cells above and stored in
`outputs/nb/07_regional_comparison/tables/regional_summary.csv`; the statements below describe the
pattern those tables show.)*

1. **Vulnerability is not evenly spread, but the gap is modest.** Under the project definition
   (critical = network-wide top-10% betweenness), the Center region has the *lowest* share of
   critical stops while the North, South and Jerusalem all sit above the 10% network average. The
   direction supports the "periphery is more fragile" hypothesis, but the spread is a few percentage
   points - not an order of magnitude.

2. **The structural signal is stronger than the traffic signal.** The share of stops that are
   articulation points differs between regions by a larger relative factor than the share of
   critical stops does, with Jerusalem clearly highest and the Center lowest. Dense grids have
   redundancy; sparse networks route everything through single stops. This is the more convincing
   evidence for the regional-inequality claim.

3. **In absolute terms the Center still dominates.** It holds far more critical stops than any other
   region simply because it holds far more stops. A national-scale disruption analysis should read
   the count panel, a fairness analysis should read the percentage panel - they point in opposite
   directions, and reporting only one of them would be misleading.

4. **Statistical significance here is nearly meaningless; effect size is not.** With ~30k stops the
   chi-square p-value is astronomically small, yet Cramer's V comes out very low. Honest reading:
   the regional differences are *real but weak*. The 95% Wilson intervals on the figures are narrow
   only because every region contains thousands of stops.

5. **No region is a small sample; check the metro table instead.** All four regions hold thousands
   of stops, so none is flagged by `MIN_REGION_N`. The flag exists mainly for the metropolitan
   breakdown and for anyone re-running with a finer geographic split - any hatched bar or
   `small_sample = True` row must not be quoted as a finding.

### Honest limitations

- **The region labels are latitude bands**, drawn in stage 01 from fixed coordinate cut-offs, not
  official administrative districts. A stop just north of a cut-off is assigned to a different
  region than its actual neighbour. Everything in this notebook inherits that approximation.
- **Betweenness is approximate** (sampled `k` sources upstream) and is computed on the largest
  connected component only, so stops on small components score 0 and can never be called critical -
  even though being on a tiny isolated component is itself a form of fragility.
- **This is a topological analysis.** A stop with high betweenness is not necessarily a stop with
  many passengers; the graph has no ridership data.
- **Nothing here is causal.** The result says the periphery's networks are structurally thinner, not
  why, and not what a specific closure would cost - notebook 05 (robustness) is where removal is
  actually simulated.